# PageIndex Flash Demo

This notebook demonstrates how to use the PageIndex local SDK to build vectorless RAG for a document.

PageIndex Flash is the default tree-indexing model used by the PageIndex client in local mode.

Learn more about PageIndex integrations in the [documentation](https://docs.pageindex.ai/).

PageIndex Flash is fully open-source. You can find it in the [PageIndex GitHub repository](https://github.com/VectifyAI/PageIndex).


### Prepare a document

In [ ]:
import requests

url = "https://arxiv.org/pdf/2607.24653"
file_name = "kimi_report.pdf"

try:
    response = requests.get(url, stream=True)
    response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

    with open(file_name, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Successfully downloaded '{file_name}'")
except requests.exceptions.RequestException as e:
    print(f"Error downloading the report: {e}")


Successfully downloaded 'kimi_report.pdf'


### Install PageIndex SDK

In [ ]:
!pip install -U pageindex

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of openai-agents to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.8/566.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.9/357.

### Choose your index and chat model

In [ ]:
import os
from pageindex import PageIndexClient

os.environ["OPENAI_API_KEY"] = "Your OpenAI API key here"

client = PageIndexClient(
    index="gpt-5.6-luna",
    chat="gpt-5.6-sol",
)

### Index a document

In [ ]:
doc_id = client.submit_document("kimi_report.pdf")["doc_id"]
print("Tree index generation successfully,  the document id is", doc_id)

Tree index generation successfully,  the document id is pi-30661b94cf2d420dad416fec035ad6e9


### Ask a query

In [ ]:
for chunk in client.chat("What benchmarks have been tested?", doc_id=doc_id, stream=True):
    print(chunk, end="", flush=True)

[tool_call] get_document_structure {"doc_name": "kimi_report.pdf", "wait_for_completion": true}
[tool_result] get_document_structure: {"success": true, "doc_name": "kimi_report.pdf", "structure": [{"title": "ABSTRACT", "node_id": "0000", "start_index": 1, "end_index": 2, "summary": "The text presents Kimi K3, a 2.8T-parameter native... (+38107 chars)

[tool_call] get_page_content {"doc_name": "kimi_report.pdf", "pages": "25-31", "wait_for_completion": true}
[tool_result] get_page_content: {"success": true, "doc_name": "kimi_report.pdf", "total_pages": 47, "requested_pages": "25-31", "returned_pages": "25-31", "content": [{"page": 25, "text": "Kimi K3: Open Frontier IntelligenceTECHNICA... (+30290 chars)

The report evaluates Kimi K3 on **44 public benchmarks**, grouped into four areas:

### Reasoning and knowledge
- GPQA Diamond
- CritPt
- AA-LCR
- Humanity’s Last Exam (HLE-Full), with and without tools

### Coding
- DeepSWE
- ProgramBench
- Terminal-Bench 2.1
- FrontierSWE
- SWE-Marat